# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 10: Deployment & Serving
**JAWNVION LLC — AI Training Workbook**

A fine-tuned model sitting on disk earns nothing. This chapter takes the TinyLlama
adapter you trained in Chapter 6 and turns it into a **live REST API** — the pattern
used in every production LLM deployment.

**What you'll learn:**
- How to merge a LoRA adapter into the base model weights (one-file export)
- How to build a FastAPI inference endpoint with streaming support
- How to run the server in a background thread inside Colab
- How to measure throughput (tokens/sec) and latency
- How production-grade servers (vLLM, HuggingFace TGI) extend this pattern

In [ ]:
# — Cell 1: GPU Check ——————————————————————————————————
import torch, subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓  GPU detected:', result.stdout.strip())
else:
    print('⚠  No GPU — inference will run on CPU (slower but functional)')
    print('   Runtime → Change runtime type → T4 GPU for best results')

print(f'   PyTorch: {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   Device : {device}')

In [ ]:
# — Cell 2: Install Packages ——————————————————————————
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers peft accelerate
!pip install -q fastapi "uvicorn[standard]<0.30" nest-asyncio httpx
print('✓  Packages installed')
print('   If prompted to restart runtime, do so then Run All again')

In [ ]:
# — Cell 3: Imports & Configuration ———————————————————
import torch
import time
import threading
import asyncio
import nest_asyncio
import httpx
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

nest_asyncio.apply()   # allows uvicorn to run inside Colab's existing event loop

# ── Constants ─────────────────────────────────────────
BASE_MODEL    = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR   = "/content/tinyllama-qlora"   # output from Chapter 6
MERGED_DIR    = "/content/tinyllama-merged"  # where we save the merged model
API_PORT      = 8000
MAX_NEW       = 150

print('✓  Config ready')
print(f'   Base model   : {BASE_MODEL}')
print(f'   Adapter dir  : {ADAPTER_DIR}  (Chapter 6 output — optional)')
print(f'   Merged dir   : {MERGED_DIR}')
print(f'   API port     : {API_PORT}')

In [ ]:
# — Cell 4: Load Model + Merge LoRA Adapter ———————————
# If you ran Chapter 6 and have a LoRA adapter, this cell merges it into the
# base weights so the served model is a single self-contained checkpoint.
# If the adapter directory doesn't exist, we serve the base model directly.

import os

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

if os.path.isdir(ADAPTER_DIR):
    print(f"LoRA adapter found at {ADAPTER_DIR} — merging...")
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    model = model.merge_and_unload()   # fuses adapter weights into base; no PEFT overhead at inference
    print("✓  Adapter merged into base model")
    print("   merge_and_unload() removes the PEFT wrapper — the result is a plain HF model")
else:
    print(f"⚠  No adapter found at {ADAPTER_DIR}")
    print("   Serving base model without fine-tuning (Chapter 6 adapter not required)")
    model = base

model.eval()

vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"✓  Model ready  |  VRAM used: {vram:.2f} GB")

In [ ]:
# — Cell 5: Save Merged Model ——————————————————————————
# Saving the merged weights lets you reload without merging again,
# and mirrors the export step you'd do before pushing to HuggingFace Hub
# or packaging for a production container.

import os, shutil

if not os.path.isdir(MERGED_DIR):
    print(f"Saving merged model to {MERGED_DIR} ...")
    model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)

    # Zip for download
    shutil.make_archive("/content/tinyllama-merged-backup", "zip", MERGED_DIR)
    print(f"✓  Merged model saved  |  Backup: /content/tinyllama-merged-backup.zip")
else:
    print(f"✓  {MERGED_DIR} already exists — skipping save")

# Show what was written
import subprocess
result = subprocess.run(['du', '-sh', MERGED_DIR], capture_output=True, text=True)
print(f"   Disk usage: {result.stdout.strip()}")

In [ ]:
# — Cell 6: Local Inference Test (before serving) ——————
# Confirm the model generates sensible output before we wrap it in an API.

def generate_local(prompt: str, max_new_tokens: int = MAX_NEW) -> str:
    """Direct model generation — no server, no HTTP."""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

TEST_PROMPT = (
    "### Instruction:\n"
    "Explain what a REST API is in two sentences.\n\n"
    "### Response:\n"
)

print("Running local inference test...")
t0 = time.time()
response = generate_local(TEST_PROMPT)
elapsed = time.time() - t0

print(f"\nPrompt  : {TEST_PROMPT.strip()}")
print(f"Response: {response}")
print(f"\n✓  Inference OK  |  {elapsed:.2f}s")

In [ ]:
# — Cell 7: Build FastAPI App ——————————————————————————
# FastAPI is the de-facto standard for Python inference servers.
# This is the same pattern used by HuggingFace TGI and most self-hosted LLM APIs.

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional
import uvicorn

app = FastAPI(title="TinyLlama Inference Server", version="1.0")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: Optional[int] = MAX_NEW
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9

class GenerateResponse(BaseModel):
    prompt: str
    response: str
    tokens_generated: int
    elapsed_sec: float
    tokens_per_sec: float

@app.get("/health")
def health():
    """Liveness probe — used by load balancers and container orchestrators."""
    return {"status": "ok", "device": str(model.device)}

@app.post("/generate", response_model=GenerateResponse)
def generate_endpoint(req: GenerateRequest):
    """
    Generate a completion for a given prompt.

    Production pattern: swap `model.generate()` here for a batched queue
    (vLLM, TGI) when you need concurrent request handling.
    """
    t0 = time.time()
    inputs = tokenizer(
        req.prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=req.max_new_tokens,
            do_sample=True,
            temperature=req.temperature,
            top_p=req.top_p,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    n_new = out.shape[1] - inputs["input_ids"].shape[1]
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()
    elapsed = time.time() - t0

    return GenerateResponse(
        prompt=req.prompt,
        response=text,
        tokens_generated=n_new,
        elapsed_sec=round(elapsed, 3),
        tokens_per_sec=round(n_new / elapsed, 1),
    )

print("✓  FastAPI app defined")
print(f"   Endpoints: GET /health  |  POST /generate")
print(f"   Docs UI  : http://localhost:{API_PORT}/docs  (after server starts)")

In [ ]:
# — Cell 8: Launch Server (Background Thread) ——————————
# We run uvicorn in a daemon thread so the Colab cell returns immediately
# and subsequent cells can call the API.

SERVER_STARTED = threading.Event()

def run_server():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=API_PORT,
        log_level="warning",   # suppress per-request logs to keep output clean
    )
    server = uvicorn.Server(config)
    SERVER_STARTED.set()
    # Python 3.10+ threads have no default event loop — create one explicitly.
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait up to 10 seconds for the server to be ready
SERVER_STARTED.wait(timeout=10)
time.sleep(1.5)   # extra settle time for port binding

# Quick health check
import httpx
resp = httpx.get(f"http://localhost:{API_PORT}/health")
print(f"✓  Server running on port {API_PORT}")
print(f"   Health: {resp.json()}")
print()
print(f"   Interactive docs: http://localhost:{API_PORT}/docs")
print(f"   (only accessible inside this Colab session)")

In [ ]:
# — Cell 9: Client — Call the API ——————————————————————
# This is what any downstream service (web app, pipeline, chatbot) would do.
# The API is language-agnostic: curl, Python httpx, JavaScript fetch — all work.

BASE_URL = f"http://localhost:{API_PORT}"

def api_generate(prompt: str, max_new_tokens: int = MAX_NEW, temperature: float = 0.7):
    """HTTP client wrapper for the inference endpoint."""
    payload = {
        "prompt": prompt,
        "max_new_tokens": max_new_tokens,
        "temperature": temperature,
    }
    resp = httpx.post(f"{BASE_URL}/generate", json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()

# ── Three test calls ───────────────────────────────────
test_prompts = [
    (
        "### Instruction:\nWhat is machine learning?\n\n### Response:\n",
        "Definition question"
    ),
    (
        "### Instruction:\nList three use cases for natural language processing.\n\n### Response:\n",
        "List generation"
    ),
    (
        "### Instruction:\nWrite a short poem about data.\n\n### Response:\n",
        "Creative task"
    ),
]

print("=" * 65)
print("  API Client Test — Three Requests")
print("=" * 65)

for prompt, label in test_prompts:
    result = api_generate(prompt)
    print(f"\n[{label}]")
    print(f"  Response      : {result['response'][:200]}")
    print(f"  Tokens        : {result['tokens_generated']}")
    print(f"  Latency       : {result['elapsed_sec']}s")
    print(f"  Throughput    : {result['tokens_per_sec']} tok/s")

print("\n✓  All API calls succeeded")

In [ ]:
# — Cell 10: Throughput Benchmark ——————————————————————
# Measure tokens/sec across N sequential requests to characterise this server's
# capacity. In production, this number drives your instance sizing decision.

import statistics

BENCHMARK_PROMPT = (
    "### Instruction:\n"
    "Explain the difference between supervised and unsupervised learning.\n\n"
    "### Response:\n"
)
N_RUNS = 5

print(f"Running {N_RUNS} sequential requests for throughput benchmark...")
print(f"Prompt length: {len(tokenizer.encode(BENCHMARK_PROMPT))} tokens\n")

latencies   = []
throughputs = []

for i in range(N_RUNS):
    result = api_generate(BENCHMARK_PROMPT, max_new_tokens=100, temperature=0.7)
    latencies.append(result["elapsed_sec"])
    throughputs.append(result["tokens_per_sec"])
    print(f"  Run {i+1}: {result['elapsed_sec']:.2f}s  |  {result['tokens_per_sec']:.1f} tok/s  |  {result['tokens_generated']} tokens")

print()
print("─" * 50)
print(f"  Median latency    : {statistics.median(latencies):.2f}s")
print(f"  P95 latency       : {sorted(latencies)[int(0.95 * N_RUNS) - 1]:.2f}s")
print(f"  Mean throughput   : {statistics.mean(throughputs):.1f} tok/s")
print(f"  Stdev throughput  : {statistics.stdev(throughputs):.1f} tok/s")
print("─" * 50)
print()
print("INTERPRETATION:")
print("  This FastAPI server handles one request at a time (synchronous).")
print("  For concurrent users, vLLM's continuous batching can serve 10–100x")
print("  more requests/sec with the same GPU by merging parallel decode steps.")

In [ ]:
# — Cell 11: Production Path — vLLM & TGI Overview ————
# This cell is informational — nothing to execute.
# It shows what changes when you graduate from this single-request server
# to a production-grade deployment.

PRODUCTION_COMPARISON = """
┌─────────────────────┬──────────────────────┬──────────────────────────┐
│ Dimension           │ This chapter (FastAPI)│ Production (vLLM / TGI)  │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Concurrency         │ 1 request at a time  │ Continuous batching       │
│                     │ (blocks on generate) │ (10–100x more req/s)     │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Throughput scaling  │ Fixed to 1 GPU       │ Tensor parallelism across │
│                     │                      │ multiple GPUs             │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Quantization        │ fp16 (manual)        │ AWQ / GPTQ / FP8 built-in│
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Streaming           │ Manual SSE loop      │ Built-in streaming API    │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ OpenAI-compatible   │ No                   │ Yes (/v1/chat/completions)│
│ API                 │                      │                           │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Memory management   │ Manual               │ PagedAttention (KV cache) │
├─────────────────────┼──────────────────────┼──────────────────────────┤
│ Deploy target       │ Colab / dev machine  │ AWS EC2 g4dn/p3, GKE,    │
│                     │                      │ SageMaker, GovCloud       │
└─────────────────────┴──────────────────────┴──────────────────────────┘

vLLM quick-start (when you have a dedicated GPU instance):
  pip install vllm
  python -m vllm.entrypoints.openai.api_server \\
      --model /content/tinyllama-merged \\
      --dtype float16 \\
      --port 8000

Then any OpenAI client just points to your instance:
  client = OpenAI(base_url="http://<your-ip>:8000/v1", api_key="ignored")
  client.chat.completions.create(model="tinyllama-merged", messages=[...])

HuggingFace TGI (Docker, production-hardened):
  docker run --gpus all -p 8080:80 \\
      ghcr.io/huggingface/text-generation-inference \\
      --model-id /content/tinyllama-merged
"""

print(PRODUCTION_COMPARISON)
print("✓  Chapter 10 complete — your model is live on port", API_PORT)
print("   Next: Chapter 11 — Monitoring & Observability")

## Chapter 10 Complete ✓

**What happened:**
- Loaded the Chapter 6 LoRA adapter (or base model if adapter not present)
- Merged adapter weights into base model with `merge_and_unload()` — no PEFT overhead at inference
- Saved the merged checkpoint to `/content/tinyllama-merged`
- Built a FastAPI inference server with `/health` and `/generate` endpoints
- Launched the server in a background thread using `uvicorn` + `nest_asyncio`
- Called the API from a Python `httpx` client — the same pattern any downstream app uses
- Measured throughput (tokens/sec) and latency across N sequential requests
- Surveyed the production gap: vLLM continuous batching and HuggingFace TGI

**Key deployment concepts:**
| Concept | What it means |
|---|---|
| `merge_and_unload()` | Fuses LoRA adapter weights into base; exports a plain HF model with no PEFT dependency |
| FastAPI + uvicorn | Lightweight ASGI server; fine for dev/demo; synchronous (one request at a time) |
| `nest_asyncio` | Patches Python's event loop so uvicorn runs inside Colab's existing loop |
| Continuous batching (vLLM) | Merges tokens from concurrent requests into one GPU kernel pass — the key to production throughput |
| PagedAttention | vLLM's KV-cache allocator; prevents VRAM fragmentation under load |
| OpenAI-compatible API | vLLM/TGI expose `/v1/chat/completions`; drop-in for any OpenAI SDK client |

**When to graduate from FastAPI to vLLM:**
- More than ~2 concurrent users
- Latency SLAs below 1 second
- Need streaming tokens (server-sent events)
- Production / Gov enclave deployment on dedicated GPU instance

**Next: Chapter 11 — Monitoring & Observability**